# 1. Objetivo

Consolidar as três tabelas e as oito figuras científicas finais sem digitar resultados manualmente. A Tabela 1 é importada do notebook de caracterização, a Tabela 3 é importada da análise principal e os demais produtos são derivados do dataset processado por funções reutilizáveis.

# 2. Importações

In [1]:
from pathlib import Path
import hashlib
import platform
import sys

import pandas as pd
from matplotlib import pyplot as plt

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.plots import (
    plot_findrisc_by_diabetes,
    plot_findrisc_by_rheumatoid_arthritis,
    plot_findrisc_categories,
    plot_imc_categories,
    plot_imc_by_diabetes,
    plot_imc_by_rheumatoid_arthritis,
    plot_imc_findrisc,
    plot_metabolic_profile,
    save_figure,
)
from src.statistics import descriptive_numeric, proportion_with_ci
from src.variables import (
    count_autoimmune_diagnoses,
    create_artrite_reumatoide_registrada,
    create_diabetes_registrado,
)

# 3. Configuração

In [2]:
PROCESSED_PATH = PROJECT_ROOT / 'data' / 'processed' / 'pacientes_clean.csv'
RAW_PATH = PROJECT_ROOT / 'data' / 'raw' / 'sociodemografico.csv'
TABLES_DIR = PROJECT_ROOT / 'outputs' / 'tables'
FIGURES_DIR = PROJECT_ROOT / 'outputs' / 'figures'
TABLE_1_PATH = TABLES_DIR / 'table_1_population.csv'
TABLE_2_PATH = TABLES_DIR / 'table_2_metabolic_profile.csv'
TABLE_3_PATH = TABLES_DIR / 'table_3_imc_findrisc.csv'
FIGURE_1_PATH = FIGURES_DIR / 'figure_1_imc_categories.png'
FIGURE_2_PATH = FIGURES_DIR / 'figure_2_findrisc_categories.png'
FIGURE_3_PATH = FIGURES_DIR / 'figure_3_imc_findrisc.png'
FIGURE_4_PATH = FIGURES_DIR / 'figure_4_metabolic_profile.png'
FIGURE_5_PATH = FIGURES_DIR / 'figure_5_imc_by_diabetes.png'
FIGURE_6_PATH = FIGURES_DIR / 'figure_6_imc_by_rheumatoid_arthritis.png'
FIGURE_7_PATH = FIGURES_DIR / 'figure_7_findrisc_by_rheumatoid_arthritis.png'
FIGURE_8_PATH = FIGURES_DIR / 'figure_8_findrisc_by_diabetes.png'
IMC_ORDER = [
    'Baixo peso', 'Peso normal', 'Sobrepeso', 'Obesidade grau I',
    'Obesidade grau II', 'Obesidade grau III',
]
FINDRISC_ORDER = ['Baixo risco', 'Leve/moderado', 'Moderado', 'Alto', 'Muito alto']
TABLE_1_TITLE = 'Características sociodemográficas e clínicas dos pacientes com doenças autoimunes'
TABLE_2_TITLE = 'Perfil antropométrico e risco metabólico dos pacientes com doenças autoimunes'
TABLE_3_TITLE = 'Associação entre índice de massa corporal e escore FINDRISC'
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f'Python: {platform.python_version()}')
print(f'pandas: {pd.__version__}')

Python: 3.14.7
pandas: 3.0.5


# 4. Carregamento

Somente as nove variáveis necessárias às tabelas e figuras são carregadas do dataset anonimizado. As Tabelas 1 e 3 são resultados agregados previamente persistidos. A comparação por diabetes é restrita aos pacientes com pelo menos uma doença autoimune explícita.

In [3]:
required_columns = [
    'imc', 'imc_categoria', 'findrisc_score', 'findrisc_categoria',
    'excesso_peso', 'obesidade', 'findrisc_alto',
    'diagnostico_padronizado', 'comorbidade_medicamento',
]
df = pd.read_csv(PROCESSED_PATH, usecols=required_columns)
df = df[required_columns]
for column in ['excesso_peso', 'obesidade', 'findrisc_alto']:
    df[column] = df[column].astype('boolean')
n_diagnosticos_autoimunes = count_autoimmune_diagnoses(df['diagnostico_padronizado'])
autoimmune_mask = n_diagnosticos_autoimunes.ge(1).fillna(False)
autoimmune = df.loc[autoimmune_mask].copy()
autoimmune['diabetes_registrado'] = create_diabetes_registrado(
    autoimmune['comorbidade_medicamento']
)
diabetes_comparison = autoimmune[['imc', 'diabetes_registrado']].dropna()
findrisc_diabetes_comparison = autoimmune[[
    'findrisc_score', 'diabetes_registrado'
]].dropna()
autoimmune['artrite_reumatoide'] = create_artrite_reumatoide_registrada(
    autoimmune['diagnostico_padronizado']
)
arthritis_comparison = autoimmune[['imc', 'artrite_reumatoide']].dropna()
findrisc_arthritis_comparison = autoimmune[[
    'findrisc_score', 'artrite_reumatoide'
]].dropna()
table_1 = pd.read_csv(TABLE_1_PATH)
table_3 = pd.read_csv(TABLE_3_PATH)
print(f'Dataset processado: {len(df)} pacientes')
print('Tabelas-fonte agregadas carregadas sem dados individualizados.')

Dataset processado: 75 pacientes
Tabelas-fonte agregadas carregadas sem dados individualizados.


# 5. Validações

A consolidação é interrompida se os arquivos anteriores, as variáveis necessárias ou a correspondência do N da análise principal falharem.

In [4]:
assert len(df) == 75
assert list(df.columns) == required_columns
assert pd.api.types.is_numeric_dtype(df['imc'])
assert pd.api.types.is_numeric_dtype(df['findrisc_score'])
assert len(autoimmune) == 69
assert autoimmune['comorbidade_medicamento'].isna().sum() == 4
assert autoimmune['diabetes_registrado'].notna().sum() == 69
assert autoimmune['diabetes_registrado'].isna().sum() == 0
assert len(diabetes_comparison) == 67
assert diabetes_comparison['diabetes_registrado'].sum() == 14
assert diabetes_comparison['diabetes_registrado'].eq(False).sum() == 53
assert len(findrisc_diabetes_comparison) == 68
assert findrisc_diabetes_comparison['diabetes_registrado'].sum() == 14
assert findrisc_diabetes_comparison['diabetes_registrado'].eq(False).sum() == 54
assert autoimmune['artrite_reumatoide'].sum() == 25
assert autoimmune['artrite_reumatoide'].eq(False).sum() == 44
assert len(arthritis_comparison) == 67
assert arthritis_comparison['artrite_reumatoide'].sum() == 23
assert arthritis_comparison['artrite_reumatoide'].eq(False).sum() == 44
assert len(findrisc_arthritis_comparison) == 68
assert findrisc_arthritis_comparison['artrite_reumatoide'].sum() == 25
assert findrisc_arthritis_comparison['artrite_reumatoide'].eq(False).sum() == 43
assert len(table_1) == 64
assert {
    'Estado civil', 'Escolaridade', 'Renda mensal familiar', 'Ocupação',
    'Diagnóstico', 'IMC (kg/m²)', 'FINDRISC (pontos)',
}.issubset(set(table_1['variavel']))
assert len(table_3) == 1
assert {'n', 'rho_spearman', 'ic95_inferior', 'ic95_superior', 'p_valor'}.issubset(
    table_3.columns
)
paired_n = int(df[['imc', 'findrisc_score']].dropna().shape[0])
assert int(table_3.loc[0, 'n']) == paired_n
print('Validações de esquema, N e proveniência: APROVADAS')
print(f'Pacientes com doença autoimune explícita: {len(autoimmune)}')
print(
    'Diabetes/pré-diabetes registrado com IMC válido: '
    f"{int(diabetes_comparison['diabetes_registrado'].sum())}"
)
print(
    'Sem diabetes/pré-diabetes registrado com IMC válido: '
    f"{int(diabetes_comparison['diabetes_registrado'].eq(False).sum())}"
)
print(
    'Diabetes/pré-diabetes registrado com FINDRISC válido: '
    f"{int(findrisc_diabetes_comparison['diabetes_registrado'].sum())}"
)
print(
    'Sem diabetes/pré-diabetes registrado com FINDRISC válido: '
    f"{int(findrisc_diabetes_comparison['diabetes_registrado'].eq(False).sum())}"
)
print('Comorbidade ausente incluída como sem diabetes registrada: 4; IMC ausente: 2')
print(
    'Pacientes com artrite reumatoide explícita: '
    f"{int(autoimmune['artrite_reumatoide'].sum())}"
)
print(
    'Artrite reumatoide com IMC válido: '
    f"{int(arthritis_comparison['artrite_reumatoide'].sum())}"
)
print(
    'Outras doenças autoimunes com IMC válido: '
    f"{int(arthritis_comparison['artrite_reumatoide'].eq(False).sum())}"
)
print(
    'Artrite reumatoide com FINDRISC válido: '
    f"{int(findrisc_arthritis_comparison['artrite_reumatoide'].sum())}"
)
print(
    'Outras doenças autoimunes com FINDRISC válido: '
    f"{int(findrisc_arthritis_comparison['artrite_reumatoide'].eq(False).sum())}"
)

Validações de esquema, N e proveniência: APROVADAS
Pacientes com doença autoimune explícita: 69
Diabetes/pré-diabetes registrado com IMC válido: 14
Sem diabetes/pré-diabetes registrado com IMC válido: 53
Diabetes/pré-diabetes registrado com FINDRISC válido: 14
Sem diabetes/pré-diabetes registrado com FINDRISC válido: 54
Comorbidade ausente incluída como sem diabetes registrada: 4; IMC ausente: 2
Pacientes com artrite reumatoide explícita: 25
Artrite reumatoide com IMC válido: 23
Outras doenças autoimunes com IMC válido: 44
Artrite reumatoide com FINDRISC válido: 25
Outras doenças autoimunes com FINDRISC válido: 43


# 6. Análise e tabelas

## Tabela 1

**Características sociodemográficas e clínicas dos pacientes com doenças autoimunes**

Esta tabela é reutilizada integralmente do notebook 02. Percentuais categóricos usam o denominador válido de cada variável.

In [5]:
table_1.to_csv(TABLE_1_PATH, index=False)
print(f'Tabela 1 validada: {len(table_1)} linhas')
table_1

Tabela 1 validada: 64 linhas


,secao,variavel,categoria,n_valido,n_ausente,n,percentual,media,moda,desvio_padrao,mediana,p25,p75,minimo,maximo,observacoes
0,Sociodemográficas,Estado civil,Solteiro,75,0,38.0,50.67,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Sociodemográficas,Estado civil,Casado,75,0,26.0,34.67,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Sociodemográficas,Estado civil,Viúvo,75,0,6.0,8.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Sociodemográficas,Estado civil,Divorciado,75,0,5.0,6.67,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Sociodemográficas,Escolaridade,Ensino fundamental incompleto,75,0,21.0,28.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,Metabólicas,Categoria FINDRISC,Baixo risco,74,1,10.0,13.51,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Classificação recalculada pela pontuação.
60,Antropométricas,IMC (kg/m²),Resumo quantitativo,73,2,NaN,NaN,28.72,29,5.50,27.94,24.62,32.3,17.5,41.1,Nenhum valor extremo foi excluído.
61,Antropométricas,Circunferência abdominal (cm),Resumo quantitativo,73,2,NaN,NaN,94.97,79; 98; 108,12.18,96.00,86.00,104.0,71.0,126.0,Distribuição multimodal (3 modas).
62,Antropométricas,Circunferência cervical (cm),Resumo quantitativo,75,0,NaN,NaN,37.36,35,9.21,36.00,34.00,39.0,30.0,110.0,Valor máximo requer conferência; nenhum valor ...


## Tabela 2

**Perfil antropométrico e risco metabólico dos pacientes com doenças autoimunes**

IMC e FINDRISC recebem estatísticas quantitativas. Excesso de peso, obesidade e FINDRISC ≥15 recebem apenas `n (%)`, com denominador válido próprio.

In [6]:
imc_summary = descriptive_numeric(df['imc'])
findrisc_summary = descriptive_numeric(df['findrisc_score'])
excesso_summary = proportion_with_ci(df['excesso_peso'])
obesidade_summary = proportion_with_ci(df['obesidade'])
findrisc_high_summary = proportion_with_ci(df['findrisc_alto'])

def quantitative_row(label, summary):
    return {
        'variavel': label,
        'n_valido': summary['n_valido'],
        'media': summary['media'],
        'moda': summary['moda'],
        'desvio_padrao': summary['desvio_padrao'],
        'mediana': summary['mediana'],
        'p25': summary['p25'],
        'p75': summary['p75'],
        'n': pd.NA,
        'percentual': pd.NA,
    }

def prevalence_row(label, summary):
    return {
        'variavel': label,
        'n_valido': summary['n_valido'],
        'media': pd.NA,
        'moda': pd.NA,
        'desvio_padrao': pd.NA,
        'mediana': pd.NA,
        'p25': pd.NA,
        'p75': pd.NA,
        'n': summary['n'],
        'percentual': summary['percentual'],
    }

table_2 = pd.DataFrame([
    quantitative_row('IMC (kg/m²)', imc_summary),
    prevalence_row('Excesso de peso (IMC ≥ 25 kg/m²)', excesso_summary),
    prevalence_row('Obesidade (IMC ≥ 30 kg/m²)', obesidade_summary),
    quantitative_row('FINDRISC (pontos)', findrisc_summary),
    prevalence_row('FINDRISC ≥ 15 pontos', findrisc_high_summary),
])
numeric_columns = [
    'media', 'desvio_padrao', 'mediana', 'p25', 'p75', 'percentual',
]
table_2[numeric_columns] = table_2[numeric_columns].apply(pd.to_numeric).round(2)
table_2.to_csv(TABLE_2_PATH, index=False)
print(f'Tabela 2 gerada: {len(table_2)} linhas')
table_2

Tabela 2 gerada: 5 linhas


,variavel,n_valido,media,moda,desvio_padrao,mediana,p25,p75,n,percentual
0,IMC (kg/m²),73,28.72,29,5.50,27.94,24.62,32.3,<NA>,NaN
1,Excesso de peso (IMC ≥ 25 kg/m²),73,NaN,NaN,NaN,NaN,NaN,NaN,52,71.23
2,Obesidade (IMC ≥ 30 kg/m²),73,NaN,NaN,NaN,NaN,NaN,NaN,26,35.62
3,FINDRISC (pontos),74,13.76,9,6.25,14.00,9.00,19.0,<NA>,NaN
4,FINDRISC ≥ 15 pontos,74,NaN,NaN,NaN,NaN,NaN,NaN,35,47.30


## Tabela 3

**Associação entre índice de massa corporal e escore FINDRISC**

A estimativa principal é reutilizada do notebook 05, incluindo IC95% por bootstrap percentil pareado com 10.000 reamostragens.

In [7]:
table_3.to_csv(TABLE_3_PATH, index=False)
main_result = table_3.iloc[0]
print(
    f"Tabela 3 validada: N = {int(main_result['n'])}; "
    f"rho = {main_result['rho_spearman']:.3f}; "
    f"IC95% {main_result['ic95_inferior']:.3f} a {main_result['ic95_superior']:.3f}; "
    f"p = {main_result['p_valor']:.3e}".replace('.', ',')
)
print('Nenhum resultado inferencial foi recalculado neste notebook.')
table_3

Tabela 3 validada: N = 72; rho = 0,704; IC95% 0,561 a 0,808; p = 5,027e-12
Nenhum resultado inferencial foi recalculado neste notebook.


,n,rho_spearman,ic95_inferior,ic95_superior,p_valor,metodo_ic,bootstrap,random_state
0,72,0.704457,0.561385,0.807711,5.026910e-12,Bootstrap percentil pareado,10000,42


# 7. Visualização

As figuras finais são produzidas pelas funções centralizadas em `src/plots.py`. A Figura 3 reutiliza o resultado persistido da análise principal; a correlação e seu intervalo de confiança não são recalculados aqui. As Figuras 5 e 6 comparam descritivamente o IMC apenas entre pacientes com doença autoimune explícita. A Figura 7 compara o FINDRISC entre artrite reumatoide e outras doenças autoimunes na mesma população restrita, e a Figura 8 compara o FINDRISC segundo o registro de diabetes ou pré-diabetes. Nas Figuras 6 e 7, o grupo de artrite reumatoide inclui todo paciente que possua esse diagnóstico, mesmo quando outra condição também foi registrada.

In [8]:
figure_1, _ = plot_imc_categories(df['imc_categoria'], order=IMC_ORDER)
save_figure(figure_1, FIGURE_1_PATH)
plt.close(figure_1)
print('Figura 1 gerada: distribuição das categorias de IMC')

figure_2, _ = plot_findrisc_categories(df['findrisc_categoria'], order=FINDRISC_ORDER)
save_figure(figure_2, FIGURE_2_PATH)
plt.close(figure_2)
print('Figura 2 gerada: distribuição das categorias FINDRISC')

figure_3, _ = plot_imc_findrisc(
    df['imc'],
    df['findrisc_score'],
    rho=main_result['rho_spearman'],
    ic95_inferior=main_result['ic95_inferior'],
    ic95_superior=main_result['ic95_superior'],
)
save_figure(figure_3, FIGURE_3_PATH)
plt.close(figure_3)
print(
    f"Figura 3 gerada: IMC × FINDRISC; N = {int(main_result['n'])}; "
    f"rho = {main_result['rho_spearman']:.3f}; "
    f"IC95% {main_result['ic95_inferior']:.3f} a "
    f"{main_result['ic95_superior']:.3f}".replace('.', ',')
)

figure_4, _ = plot_metabolic_profile(
    df['excesso_peso'], df['obesidade'], df['findrisc_alto']
)
save_figure(figure_4, FIGURE_4_PATH)
plt.close(figure_4)
print('Figura 4 gerada: perfil metabólico geral')

figure_5, _ = plot_imc_by_diabetes(
    autoimmune['imc'], autoimmune['diabetes_registrado']
)
save_figure(figure_5, FIGURE_5_PATH)
plt.close(figure_5)
print(f'Figura 5 gerada: IMC por diabetes registrado; N = {len(diabetes_comparison)}')

figure_6, _ = plot_imc_by_rheumatoid_arthritis(
    autoimmune['imc'], autoimmune['artrite_reumatoide']
)
save_figure(figure_6, FIGURE_6_PATH)
plt.close(figure_6)
print(f'Figura 6 gerada: IMC por artrite reumatoide; N = {len(arthritis_comparison)}')

figure_7, _ = plot_findrisc_by_rheumatoid_arthritis(
    autoimmune['findrisc_score'], autoimmune['artrite_reumatoide']
)
save_figure(figure_7, FIGURE_7_PATH)
plt.close(figure_7)
print(
    'Figura 7 gerada: FINDRISC por artrite reumatoide; '
    f'N = {len(findrisc_arthritis_comparison)}'
)

figure_8, _ = plot_findrisc_by_diabetes(
    autoimmune['findrisc_score'], autoimmune['diabetes_registrado']
)
save_figure(figure_8, FIGURE_8_PATH)
plt.close(figure_8)
print(
    'Figura 8 gerada: FINDRISC por diabetes registrado; '
    f'N = {len(findrisc_diabetes_comparison)}'
)
print('Oito figuras finais exportadas em PNG a 300 DPI.')

Figura 1 gerada: distribuição das categorias de IMC


Figura 2 gerada: distribuição das categorias FINDRISC


Figura 3 gerada: IMC × FINDRISC; N = 72; rho = 0,704; IC95% 0,561 a 0,808


Figura 4 gerada: perfil metabólico geral


Figura 5 gerada: IMC por diabetes registrado; N = 67


Figura 6 gerada: IMC por artrite reumatoide; N = 67


Figura 7 gerada: FINDRISC por artrite reumatoide; N = 68


Figura 8 gerada: FINDRISC por diabetes registrado; N = 68
Oito figuras finais exportadas em PNG a 300 DPI.


# 8. Conclusões deste notebook

As três tabelas e as oito figuras finais possuem proveniência reproduzível. Nenhuma média ou desvio-padrão foi calculado para os indicadores categóricos; missing permaneceu fora dos denominadores válidos, exceto pela regra explícita usada nas Figuras 5 e 8. Nessas figuras, o grupo `diabetes registrado` inclui pré-diabetes por decisão analítica; pacientes autoimunes sem informação no campo de comorbidades integram `sem diabetes registrado`, que não equivale à confirmação clínica de ausência da doença. As Figuras 6 e 7 também são exploratórias e não isolam o efeito do diagnóstico de artrite reumatoide de outras características clínicas. As figuras não apresentam nomes nem identificadores de pacientes.

In [9]:
categorical_rows = table_2['n'].notna()
quantitative_columns = ['media', 'moda', 'desvio_padrao', 'mediana', 'p25', 'p75']
assert table_2.loc[categorical_rows, quantitative_columns].isna().all().all()
assert table_2.loc[~categorical_rows, ['n', 'percentual']].isna().all().all()
assert set(diabetes_comparison['diabetes_registrado'].astype(bool)) == {False, True}
assert set(findrisc_diabetes_comparison['diabetes_registrado'].astype(bool)) == {False, True}
assert set(arthritis_comparison['artrite_reumatoide'].astype(bool)) == {False, True}
assert set(findrisc_arthritis_comparison['artrite_reumatoide'].astype(bool)) == {False, True}
print('Separação entre estatísticas quantitativas e n (%): APROVADA')
print('Comparação de diabetes restrita aos pacientes autoimunes: APROVADA')
print('Comparação de artrite reumatoide restrita aos pacientes autoimunes: APROVADA')
print('Todos os números foram derivados do dataset processado ou de resultados agregados anteriores.')

Separação entre estatísticas quantitativas e n (%): APROVADA
Comparação de diabetes restrita aos pacientes autoimunes: APROVADA
Comparação de artrite reumatoide restrita aos pacientes autoimunes: APROVADA
Todos os números foram derivados do dataset processado ou de resultados agregados anteriores.


# 9. Outputs gerados

In [10]:
table_paths = [TABLE_1_PATH, TABLE_2_PATH, TABLE_3_PATH]
figure_paths = [
    FIGURE_1_PATH, FIGURE_2_PATH, FIGURE_3_PATH, FIGURE_4_PATH,
    FIGURE_5_PATH, FIGURE_6_PATH, FIGURE_7_PATH, FIGURE_8_PATH,
]
output_paths = table_paths + figure_paths
assert all(path.exists() and path.stat().st_size > 0 for path in output_paths)
assert hashlib.sha256(RAW_PATH.read_bytes()).hexdigest() == (
    'a895b5340c856d2cd1772c8e115ed5efc20f409d3aef5ecba14da23d44da9bf4'
)
print('Tabelas finais geradas e validadas:')
for path in table_paths:
    print(f'- {path.relative_to(PROJECT_ROOT)}')
print('Figuras finais geradas e validadas:')
for path in figure_paths:
    print(f'- {path.relative_to(PROJECT_ROOT)}')
print('Integridade do arquivo bruto confirmada por SHA-256.')

Tabelas finais geradas e validadas:
- outputs/tables/table_1_population.csv
- outputs/tables/table_2_metabolic_profile.csv
- outputs/tables/table_3_imc_findrisc.csv
Figuras finais geradas e validadas:
- outputs/figures/figure_1_imc_categories.png
- outputs/figures/figure_2_findrisc_categories.png
- outputs/figures/figure_3_imc_findrisc.png
- outputs/figures/figure_4_metabolic_profile.png
- outputs/figures/figure_5_imc_by_diabetes.png
- outputs/figures/figure_6_imc_by_rheumatoid_arthritis.png
- outputs/figures/figure_7_findrisc_by_rheumatoid_arthritis.png
- outputs/figures/figure_8_findrisc_by_diabetes.png
Integridade do arquivo bruto confirmada por SHA-256.
